In [ ]:
import os
from pathlib import Path

import re
from ultralytics import YOLO
import pandas as pd
import pandas as pd
import cv2
from PIL import Image


Execution du modèle YOLO

In [ ]:
def predict(chosen_model, img, classes=[], conf=0.5):
    if classes:
        results = chosen_model.predict(img, classes=classes, conf=conf)
    else:
        results = chosen_model.predict(img, conf=conf)

    return results


def predict_and_detect(chosen_model, img, classes=[], conf=0.5, rectangle_thickness=5, text_thickness=1):
    results = predict(chosen_model, img, classes, conf=conf)
    for result in results:
        for box in result.boxes:
            cv2.rectangle(img, (int(box.xyxy[0][0]), int(box.xyxy[0][1])),
                          (int(box.xyxy[0][2]), int(box.xyxy[0][3])), (255, 0, 255), rectangle_thickness)
            
    return img, results


            

In [ ]:
image_folder = "" # indiquer ici le nom du dossier contenant les images à traiter  
model_name = "" #  nom du modèle à utiliser, stocké au même endroit que le notebook = ex: "best.pt" 
image_type = "" #indiquer le type d'images à traiter ("".jpeg", ".png"...etc) 

In [ ]:
#Commenter ou décommenter les dossiers supplémentaires selon les outputs souhaités

# os.makedirs("textes", exist_ok =True)
os.makedirs("csvs", exist_ok=True)
# os.makedirs("images", exist_ok=True)
os.makedirs("crops", exist_ok=True)
model = YOLO(model_name)

# Définir l'index de la ou les classes à détecter (ex: 0 pour la première classe). Les classes pour un modèle YOLO sont contenues dans model.names
TARGET_CLASS = 7

path = Path.cwd()/"echantillon9"

for root, dirs, files in os.walk(path):
    for f in files:
        if f.endswith(image_type): #modifier selon la nature de nos images  
            image_path = os.path.join(root, f)
            image = cv2.imread(image_path)

            if image is None:
                print(f"Impossible de lire l'image : {image_path}")
                continue

            result_img, results = predict_and_detect(model, image, classes=[TARGET_CLASS], conf=0.5)

            for result in results:
                # Filtrer les détections pour ne garder que la classe cible
                filtered_boxes = result.boxes[result.boxes.cls == TARGET_CLASS]
                result.boxes = filtered_boxes

                
                csv_path = f"csvs/{f.replace(image_type,"")}_annots.csv"
                csv_content = result.to_csv()
                if len(result.boxes)>0:
                    print(csv_content)
                    with open(csv_path, "w") as kk:
                        kk.write(csv_content)  

                # Décommenter pour les coordonnées en fichier texte
               # with open(f"textes/coord_{f.replace(image_type,"")}.txt", 'a') as txt_file:
               #     for box in result.boxes:
               #         txt_file.write(str(box) + "\n")

            # Décommenter pour afficher la visualisation
            # cv2.imshow("Image", result_img)
            # cv2.waitKey(0)
            #Décommenter pour enregistrer l'image avec les BBOX tracées 
            # cv2.imwrite(f"images/result_{f.replace(image_type,"")}.png", result_img)
            

Extraction des crops des résultats

In [ ]:
def extract_images(fichier, chemin_image):
    df=pd.read_csv(f"./csvs/{fichier}")
    if not df.empty:
        liste_coords =df['box'].to_list()
            
        path=os.getcwd()
        
        image = cv2.imread(path + "/"+ chemin_image)
        
        if image is None:
                print(f"Erreur : impossible de charger l'image {chemin_image}")
                exit()
        nom_dossier = (chemin_image.split("/")[-1]).replace(image_type,"") #modifier selon le format d'image 
        print(nom_dossier)
        if not os.path.exists(path+"/"+"crops"+"/"+nom_dossier):
            os.makedirs("crops/"+nom_dossier)
            print(f"{nom_dossier} créé")
        compteur=0
        for i in liste_coords:
            compteur+=1
            dico = ast.literal_eval(i)
            valeurs=list(dico.values())
            
            y1, x1, y2, x2 = int(round(valeurs[0])), int(round(valeurs[1])), int(round(valeurs[2])), int(round(valeurs[3]))
            cropped_img= image[x1:x2,y1:y2]
            cv2.imwrite(f"crops/{nom_dossier}/{nom_dossier}_illus{compteur}.png", cropped_img)

In [ ]:

path=os.getcwd()+"/"+image_folder
for root, dirs, file in os.walk(path):
    for image in file:
        if image_type in image:
            
                nom_csv = image.replace(image_type,"")+"_annots.csv"
                if os.path.isfile(f"csvs/{nom_csv}"):
                    chemin_image = image_folder +"/"+image
                    extract_images(nom_csv, chemin_image)
            
